<a href="https://colab.research.google.com/github/DeshGargi/DeshGargi.github.io/blob/main/vector_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 - INSTALL DEPENDENCIES

In [ ]:
!pip install pdfplumber pymupdf --quiet

In [ ]:
from google.colab import files
import os

# This will open a file picker dialog
uploaded = files.upload()

# Grab the filename automatically
PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")

# Output folder — stored in Colab's local session storage (not Drive)
IMAGE_OUT_DIR = '/content/extracted_images'
os.makedirs(IMAGE_OUT_DIR, exist_ok=True)
print(f"Images will save to: {IMAGE_OUT_DIR}")

3 - EXTRACT TEXT AND TABLES

In [ ]:
import pdfplumber
import json

pages_data = []

with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages):

        raw_text = page.extract_text()
        tables = page.extract_tables()

        parsed_tables = []
        for table in tables:
            if not table:
                continue
            headers = table[0]
            rows = []
            for row in table[1:]:
                rows.append(dict(zip(headers, row)))
            parsed_tables.append(rows)

        page_record = {
            'page': page_num + 1,
            'text': raw_text,
            'tables': parsed_tables,
        }
        pages_data.append(page_record)

        print(f"--- Page {page_num + 1} ---")
        print(f"  Text length: {len(raw_text) if raw_text else 0} chars")
        print(f"  Tables found: {len(parsed_tables)}")
        if parsed_tables:
            for i, t in enumerate(parsed_tables):
                print(f"    Table {i+1}: {len(t)} rows, columns: {list(t[0].keys()) if t else '?'}")

3B - FIX TABLE EXTRACTION WITH A FALL BACK

In [ ]:
# For pages where table parsing failed, extract raw text from the table region
# and store it as a plain string instead

with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        record = pages_data[i]

        # If tables were detected but came back malformed (0 rows or bad headers)
        raw_tables = page.extract_tables()
        fixed_tables = []

        for table in raw_tables:
            if not table or len(table) < 2:
                # Store raw table as text fallback
                raw = page.extract_text_simple() if hasattr(page, 'extract_text_simple') else page.extract_text()
                fixed_tables.append({'raw_fallback': raw})
                continue

            # Check if header looks malformed (contains newlines — merged cells)
            headers = table[0]
            if any('\n' in str(h) for h in headers if h):
                # Flatten: treat entire table as text
                flat = '\n'.join(['\t'.join([str(c) for c in row if c]) for row in table])
                fixed_tables.append({'raw_fallback': flat})
            else:
                # Normal parse
                rows = [dict(zip(headers, row)) for row in table[1:]]
                fixed_tables.append({'rows': rows})

        record['tables'] = fixed_tables

print("Table re-extraction complete.")
for p in pages_data:
    print(f"  Page {p['page']}: {len(p['tables'])} table(s)")
    for t in p['tables']:
        if 'raw_fallback' in t:
            print(f"    → stored as raw text ({len(t['raw_fallback'])} chars)")
        else:
            print(f"    → parsed: {len(t['rows'])} rows")

3C - PAGE 4 TABLE INSPECTION

In [ ]:
# Inspect what we actually got for page 4 tables
for t in pages_data[3]['tables']:  # index 3 = page 4
    print(repr(t['raw_fallback']))
    print("---")

In [ ]:
print(pages_data[3]['text'])

4 - EXTRACT IMAGES

In [ ]:
import fitz
from PIL import Image
import io

image_records = []

doc = fitz.open(PDF_PATH)

for page_num in range(len(doc)):
    page = doc[page_num]
    image_list = page.get_images(full=True)

    print(f"Page {page_num + 1}: {len(image_list)} image(s) found")

    for img_index, img_info in enumerate(image_list):
        xref = img_info[0]

        base_image = doc.extract_image(xref)
        img_bytes = base_image["image"]
        img_ext = base_image["ext"]

        img = Image.open(io.BytesIO(img_bytes))
        width, height = img.size
        if width < 50 or height < 50:
            print(f"  Skipping tiny image ({width}x{height})")
            continue

        filename = f"page{page_num+1}_img{img_index+1}.{img_ext}"
        save_path = os.path.join(IMAGE_OUT_DIR, filename)
        with open(save_path, 'wb') as f:
            f.write(img_bytes)

        image_records.append({
            'filename': filename,
            'path': save_path,
            'page': page_num + 1,
            'width': width,
            'height': height,
            'format': img_ext,
            'caption': None
        })
        print(f"  Saved: {filename} ({width}x{height})")

doc.close()
print(f"\nTotal images extracted: {len(image_records)}")

4B - DEDUPLICATE IMAGES BY SIZE

In [ ]:
# Deduplicate: keep only one copy when two images have identical dimensions
seen_sizes = {}
deduplicated = []

for record in image_records:
    key = (record['page'], record['width'], record['height'])
    if key not in seen_sizes:
        seen_sizes[key] = True
        deduplicated.append(record)
    else:
        print(f"Dropping duplicate: {record['filename']} ({record['width']}x{record['height']})")

image_records = deduplicated
print(f"\nAfter dedup: {len(image_records)} images")

5 - SANITY CHECK

In [ ]:
from IPython.display import display
from PIL import Image

if image_records:
    sample = image_records[0]
    print(f"Showing: {sample['filename']} from page {sample['page']}")
    display(Image.open(sample['path']))
else:
    print("No images were extracted — check the skip threshold or PDF structure.")

5A - CHECK WHAT THE 26 IMAGES ARE

In [ ]:
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt
import math

cols = 4
rows = math.ceil(len(image_records) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
axes = axes.flatten()

for i, record in enumerate(image_records):
    img = Image.open(record['path'])
    axes[i].imshow(img)
    axes[i].set_title(f"{record['filename']}\nPage {record['page']} | {record['width']}x{record['height']}", fontsize=8)
    axes[i].axis('off')

# Hide any unused subplot slots
for j in range(len(image_records), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

5B - REMOVE THE 2 BAD IMAGES

In [ ]:
skip_filenames = {'page1_img5.jpx', 'page1_img8.jpx'}

caption_targets = [r for r in image_records if r['filename'] not in skip_filenames]
print(f"Images to caption: {len(caption_targets)} of {len(image_records)}")

5C - CHECK ALL THE 24 IMAGES

In [ ]:
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt
import math

cols = 4
rows = math.ceil(len(caption_targets) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
axes = axes.flatten()

for i, record in enumerate(caption_targets):
    img = Image.open(record['path'])
    axes[i].imshow(img)
    axes[i].set_title(f"{record['filename']}\nPage {record['page']} | {record['width']}x{record['height']}", fontsize=8)
    axes[i].axis('off')

for j in range(len(caption_targets), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

6 - SAVE CHECKPOINT

In [ ]:
CHECKPOINT_PATH = '/content/parsed_pages.json'
with open(CHECKPOINT_PATH, 'w') as f:
    json.dump(pages_data, f, indent=2, default=str)

print(f"Saved {len(pages_data)} pages to {CHECKPOINT_PATH}")

In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get("KG_summer'26_project")
print("Key loaded:", GEMINI_API_KEY[:8] + "...")

In [ ]:
!pip install -q google-genai

In [ ]:
for record in caption_targets:
    record['caption'] = f"Technical diagram from page {record['page']} of Danfoss EZ Clip assembly manual. Filename: {record['filename']}. Dimensions: {record['width']}x{record['height']}."

print("Placeholder captions added.")
print(f"Sample: {caption_targets[0]['caption']}")

7 - INSTALL EMBEDDING DEPENDENCIES

In [ ]:
!pip install sentence-transformers chromadb --quiet

8 - CHUNK THE TEXT

In [ ]:
def chunk_text(text, chunk_size=400, overlap=50):
    """Split text into overlapping word-level chunks."""
    if not text:
        return []

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # slide forward with overlap

    return chunks

# Build a flat list of all chunks with metadata
text_chunks = []
chunk_id = 0

for page in pages_data:
    # Chunk the main page text
    chunks = chunk_text(page['text'])
    for chunk in chunks:
        text_chunks.append({
            'id': f"text_chunk_{chunk_id}",
            'content': chunk,
            'type': 'text',
            'page': page['page'],
            'source': 'page_text'
        })
        chunk_id += 1

    # Also add table content as its own chunk
    for t in page['tables']:
        if 'raw_fallback' in t and t['raw_fallback']:
            text_chunks.append({
                'id': f"text_chunk_{chunk_id}",
                'content': t['raw_fallback'],
                'type': 'table',
                'page': page['page'],
                'source': 'table'
            })
            chunk_id += 1

print(f"Total text chunks: {len(text_chunks)}")
for c in text_chunks:
    print(f"  [{c['id']}] page {c['page']} | {c['type']} | {len(c['content'].split())} words")

9 - GENERATE EMBEDDINGS WITH PLACEHOLDERS [NO CAPTIONS]

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model — downloads ~90MB on first run
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

# Embed text chunks
print("\nEmbedding text chunks...")
for chunk in text_chunks:
    chunk['embedding'] = embedder.encode(chunk['content']).tolist()
print(f"Done — {len(text_chunks)} text chunks embedded.")

# Embed image captions
print("\nEmbedding image captions...")
for record in caption_targets:
    record['embedding'] = embedder.encode(record['caption']).tolist()
print(f"Done — {len(caption_targets)} image captions embedded.")

# Sanity check
print(f"\nEmbedding dimension: {len(text_chunks[0]['embedding'])}")

10 - SET UP CHROMA & STORE EVERYTHING

In [ ]:
import chromadb

# Create an in-memory Chroma client
chroma_client = chromadb.Client()

# Create a single collection for all content
collection = chroma_client.create_collection(
    name="danfoss_ezclip",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text embeddings
)

# --- Add text chunks ---
collection.add(
    ids=[c['id'] for c in text_chunks],
    embeddings=[c['embedding'] for c in text_chunks],
    documents=[c['content'] for c in text_chunks],
    metadatas=[{
        'type': c['type'],
        'page': c['page'],
        'source': c['source']
    } for c in text_chunks]
)
print(f"Added {len(text_chunks)} text chunks to collection.")

# --- Add image captions ---
collection.add(
    ids=[r['filename'] for r in caption_targets],
    embeddings=[r['embedding'] for r in caption_targets],
    documents=[r['caption'] for r in caption_targets],
    metadatas=[{
        'type': 'image',
        'page': r['page'],
        'source': 'image_caption',
        'filename': r['filename'],
        'width': r['width'],
        'height': r['height']
    } for r in caption_targets]
)
print(f"Added {len(caption_targets)} image captions to collection.")

print(f"\nTotal items in collection: {collection.count()}")

11 - TEST A QUERY

In [ ]:
def query_collection(question, n_results=3):
    """Query the vector database with a natural language question."""
    query_embedding = embedder.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )

    print(f"Query: '{question}'\n")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"Result {i+1} | score: {1-dist:.3f} | type: {meta['type']} | page: {meta['page']}")
        print(f"  {doc[:150]}...")
        print()

# Test with a few questions
query_collection("What are the torque values for dash size 8?")
query_collection("How do I insert the nipple into the hose?")
query_collection("What tool is used to close the clips?")

12 - CONNECTING TO GEMINI TO GET BETTER CAPTIONS FOR THE IMAGES

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("KG_summer'26_project")
client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)
print(response.text)

13 - GENERATING CAPTIONS FOR THE FIRST 20 IMAGES THROUGH GEMINI

In [ ]:
import time
import base64

def encode_image(image_path):
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def get_caption(client, record):
    ext = record['format'].lower()
    mime_map = {'png': 'image/png', 'jpeg': 'image/jpeg', 'jpg': 'image/jpeg', 'jpx': 'image/jpeg'}
    mime_type = mime_map.get(ext, 'image/jpeg')

    image_data = encode_image(record['path'])

    prompt = """You are analyzing a technical image from a Danfoss EZ Clip to 5400 Series
hose assembly manual. Describe what you see in 2-3 sentences, focusing on:
- What component or assembly step is shown
- Any visible part numbers, labels, or measurements
- The purpose of this image in the context of hose assembly instructions
Be specific and technical. Do not say 'the image shows' — just describe directly."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            {
                "parts": [
                    {"inline_data": {"mime_type": mime_type, "data": image_data}},
                    {"text": prompt}
                ]
            }
        ]
    )
    return response.text

print(f"Captioning {len(caption_targets)} images...\n")

for i, record in enumerate(caption_targets):
    try:
        caption = get_caption(client, record)
        record['caption'] = caption
        print(f"[{i+1}/{len(caption_targets)}] {record['filename']}: {caption[:80]}...")
        time.sleep(4)

    except Exception as e:
        print(f"[{i+1}/{len(caption_targets)}] FAILED {record['filename']}: {e}")
        record['caption'] = None
        time.sleep(10)

print("\nDone!")

13A - SAVE THE CAPTIONS GENERATED BY GEMINI IN A JSON FILE

In [ ]:
# Save captions to JSON so we don't lose them on session restart
import json
captions_backup = [{'filename': r['filename'], 'caption': r['caption']} for r in caption_targets]
with open('/content/captions_backup.json', 'w') as f:
    json.dump(captions_backup, f, indent=2)
print("Captions saved.")

In [ ]:
import json
captions_backup = [{'filename': r['filename'], 'caption': r['caption']} for r in caption_targets]
with open('/content/captions_backup.json', 'w') as f:
    json.dump(captions_backup, f, indent=2)
print("Captions saved.")
print(f"Real captions: {sum(1 for r in caption_targets if r['caption'] and 'Technical diagram' not in r['caption'])}")
print(f"Placeholders/None: {sum(1 for r in caption_targets if not r['caption'] or 'Technical diagram' in r['caption'])}")

In [ ]:
from google.colab import files
files.download('/content/captions_backup.json')

14 - LOAD CAPTIONS FROM BACKUP INSTEAD OF CALLING GEMINI

In [ ]:
from google.colab import files
uploaded = files.upload()  # select captions_backup.json from your computer

In [ ]:
import json
with open('/content/captions_backup.json', 'r') as f:
    saved_captions = json.load(f)

caption_map = {item['filename']: item['caption'] for item in saved_captions}

for record in caption_targets:
    if record['filename'] in caption_map and caption_map[record['filename']]:
        record['caption'] = caption_map[record['filename']]

print(f"Real captions loaded: {sum(1 for r in caption_targets if r['caption'] and 'Technical diagram' not in r['caption'])}")
print(f"Still placeholder/None: {sum(1 for r in caption_targets if not r['caption'] or 'Technical diagram' in r['caption'])}")

In [ ]:
import time
import base64

def encode_image(image_path):
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def get_caption(client, record):
    ext = record['format'].lower()
    mime_map = {'png': 'image/png', 'jpeg': 'image/jpeg', 'jpg': 'image/jpeg', 'jpx': 'image/jpeg'}
    mime_type = mime_map.get(ext, 'image/jpeg')

    image_data = encode_image(record['path'])

    prompt = """You are analyzing a technical image from a Danfoss EZ Clip to 5400 Series
hose assembly manual. Describe what you see in 2-3 sentences, focusing on:
- What component or assembly step is shown
- Any visible part numbers, labels, or measurements
- The purpose of this image in the context of hose assembly instructions
Be specific and technical. Do not say 'the image shows' — just describe directly."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            {
                "parts": [
                    {"inline_data": {"mime_type": mime_type, "data": image_data}},
                    {"text": prompt}
                ]
            }
        ]
    )
    return response.text

print("Functions defined.")

14A - RE-TRY CAPTIONING FOR THE REMAINING 4 IMAGES

In [ ]:
# Retry captioning for failed images
failed = [r for r in caption_targets if not r['caption'] or 'Technical diagram' in r['caption']]
print(f"Retrying {len(failed)} failed images...\n")

for i, record in enumerate(failed):
    try:
        caption = get_caption(client, record)
        record['caption'] = caption
        print(f"[{i+1}/{len(failed)}] {record['filename']}: {caption[:80]}...")
        time.sleep(15)
    except Exception as e:
        print(f"[{i+1}/{len(failed)}] FAILED again: {e}")
        record['caption'] = None
        time.sleep(60)

print("\nDone!")

14B - SAVE ALL 24 CAPTIONS IN A JSON FILE

In [ ]:
# Save complete captions backup
import json
captions_backup = [{'filename': r['filename'], 'caption': r['caption']} for r in caption_targets]
with open('/content/captions_backup.json', 'w') as f:
    json.dump(captions_backup, f, indent=2)

print(f"Saved {len(captions_backup)} captions.")
print(f"Real captions: {sum(1 for r in caption_targets if r['caption'] and 'Technical diagram' not in r['caption'])}")
print(f"Placeholders: {sum(1 for r in caption_targets if not r['caption'] or 'Technical diagram' in r['caption'])}")

In [ ]:
from google.colab import files
files.download('/content/captions_backup.json')

15 - RE-EMBED ALL 24 IMAGES WITH THE GEMINI CAPTIONS

In [ ]:
# Re-embed with all 24 real captions + rebuild Chroma
print("Re-embedding image captions with real Gemini captions...")
for record in caption_targets:
    record['embedding'] = embedder.encode(record['caption']).tolist()
print("Done.")

chroma_client.delete_collection("danfoss_ezclip")
collection = chroma_client.create_collection(
    name="danfoss_ezclip",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[c['id'] for c in text_chunks],
    embeddings=[c['embedding'] for c in text_chunks],
    documents=[c['content'] for c in text_chunks],
    metadatas=[{'type': c['type'], 'page': c['page'], 'source': c['source']} for c in text_chunks]
)

collection.add(
    ids=[r['filename'] for r in caption_targets],
    embeddings=[r['embedding'] for r in caption_targets],
    documents=[r['caption'] for r in caption_targets],
    metadatas=[{'type': 'image', 'page': r['page'], 'source': 'image_caption', 'filename': r['filename']} for r in caption_targets]
)

print(f"Collection rebuilt with {collection.count()} items.")

16 - RE-RUN QUERY TESTS TO CHECK SCORES

In [ ]:
# Re-run query tests to compare scores
query_collection("What are the torque values for dash size 8?")
query_collection("How do I insert the nipple into the hose?")
query_collection("What tool is used to close the clips?")